# Metis: Solar Orbiter VL+UV Coronal Imager — Implementation / 구현

**Paper**: Antonucci, E., Romoli, M., Andretta, V., Fineschi, S., et al., *Metis: the Solar Orbiter visible light and ultraviolet coronal imager*, A&A 642, A10 (2020). DOI: [10.1051/0004-6361/201935338](https://doi.org/10.1051/0004-6361/201935338)

**English.** This notebook reproduces the four foundational diagnostics that Metis enables: (1) externally occulted coronagraph geometry covering 1.7–9 R☉ depending on heliocentric distance, (2) the simultaneous visible-light polarised-brightness (pB) channel and the H I Lyman-α UV channel separated by an Al/MgF₂ multilayer beam-splitter, (3) Thomson-scattering electron density retrieval $n_e(r)$ from $pB(\rho)$ via the van de Hulst inversion, and (4) inverted Doppler-dimming of resonantly scattered Lyman-α to extract the proton outflow speed $w$. All data are synthetic and the code uses NumPy/SciPy/Matplotlib only.

**한국어.** 본 노트북은 Metis가 가능케 하는 네 가지 핵심 진단을 재현한다: (1) 위성 일심거리에 따라 1.7–9 R☉ 범위를 덮는 외부 차폐형 코로나그래프 기하, (2) Al/MgF₂ 다층막 빔 스플리터로 분리되는 가시광 편광 밝기(pB) 채널과 H I Lyman-α UV 채널, (3) van de Hulst 역해를 통한 $pB(\rho)$ → $n_e(r)$ 톰슨 산란 전자 밀도 도출, (4) 공명 산란 Lyman-α의 도플러 디밍 역해로 양성자 유출 속도 $w$ 추출. 모든 데이터는 합성 데이터이며 NumPy/SciPy/Matplotlib만 사용한다.

In [ ]:
"""Imports and physical constants for Metis diagnostics.

All units are CGS unless noted otherwise. Wavelengths in angstroms,
speeds in km/s, distances in solar radii.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, brentq
from scipy.integrate import quad
from scipy.special import gamma as gamma_fn

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Physical constants (CGS).
SIGMA_T = 6.6524587e-25  # Thomson cross section [cm^2]
K_B = 1.380649e-16       # Boltzmann constant [erg/K]
M_P = 1.67262192e-24     # Proton mass [g]
C_LIGHT = 2.99792458e5   # Speed of light [km/s]
R_SUN_CM = 6.957e10      # Solar radius [cm]
AU_RSUN = 215.032        # 1 AU in solar radii
B_SUN = 2.5e10           # Mean disk brightness [erg s^-1 cm^-2 sr^-1] (mean photospheric)
U_LIMB = 0.63            # Limb-darkening coefficient at 600 nm (Allen 2000)
LAMBDA_LYA = 1215.67     # H I Lyman-alpha rest wavelength [angstrom]

## Part 1: Externally Occulted Coronagraph Geometry / 외부 차폐형 코로나그래프 기하

**English.** Metis uses an **inverted external occulter (IEO)**: a 40-mm circular aperture located 800 mm in front of the optical head defines the entrance, while an on-axis spherical mirror M0 back-reflects the disk light through that same aperture. The coronal field of view is delimited by

- inner FoV at 1.6° (set by internal occulter + Lyot stop vignetting),
- outer FoV at ±2.9° square (detector edge),
- corner FoV at 3.4° (field stop).

Because Solar Orbiter is on an eccentric orbit (perihelion 0.28 AU, aphelion ~0.9 AU), the heliocentric distance corresponding to a fixed angular radius scales linearly with the spacecraft–Sun distance $d$ (in AU): $r_\text{FoV}\,[R_\odot] = \theta\,[\text{rad}]\cdot d\,[\text{AU}]\cdot 215$.

**한국어.** Metis는 **역전 외부 차폐(IEO)** 를 채택했다: 광학 헤드 800 mm 앞에 위치한 40 mm 원형 입구가 외부 차폐로 작용하며, 광축 위 구면 거울 M0가 디스크 광을 같은 입구를 통해 우주로 후방 반사한다. 코로나 시야는 (i) 1.6° 내경(내부 차폐 + Lyot stop 비네팅), (ii) ±2.9° 정사각 외경(검출기 가장자리), (iii) 3.4° 코너(field stop)로 구획된다. Solar Orbiter는 이심 궤도(근일점 0.28 AU, 원일점 ~0.9 AU)에 있으므로, 고정 각반경에 대응하는 일심거리는 위성-태양 거리 $d$(AU)에 선형 비례한다.

In [ ]:
def fov_to_rsun(angle_deg, d_au):
    """Convert an angular FoV radius to heliocentric distance in solar radii.

    Args:
        angle_deg: Half-angle from Sun centre [deg].
        d_au: Spacecraft–Sun distance [AU].

    Returns:
        Heliocentric distance corresponding to the FoV edge [R_sun].
    """
    return np.deg2rad(angle_deg) * d_au * AU_RSUN


# Reproduce Table 5 of Antonucci et al. 2020 for several orbital phases.
phases_au = np.array([0.28, 0.40, 0.50, 0.70, 0.90])
fov_edges_deg = {'inner (1.6°)': 1.6, 'outer (2.9°)': 2.9, 'corner (3.4°)': 3.4}

print(f"{'d [AU]':>8} | " + " | ".join(f"{k:>14}" for k in fov_edges_deg))
print("-" * 70)
for d in phases_au:
    cells = [f"{fov_to_rsun(theta, d):>14.2f}" for theta in fov_edges_deg.values()]
    print(f"{d:>8.2f} | " + " | ".join(cells))

In [ ]:
"""Visualise the IEO geometry and the resulting annular field of view.

The diagram is schematic: it shows the IEO acting as the small entrance
aperture, M0 sending disk light back through it, and the coronal rays
grazing past M0 to reach the Gregorian (M1, M2).
"""
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left panel: schematic optical layout (z is the optical axis).
ax = axes[0]
ax.set_title('IEO inverted external occulter — schematic / IEO 역전 외부 차폐 개념도')
ax.axhline(0, color='gray', lw=0.5, ls='--')
# IEO aperture at z=0
ax.plot([0, 0], [-2.0, -0.2], color='black', lw=4)
ax.plot([0, 0], [0.2, 2.0], color='black', lw=4)
ax.text(0, 2.2, 'IEO (40 mm)', ha='center', fontsize=10)
# M0 mirror at z=80 cm
ax.plot([8, 8], [-0.36, 0.36], color='C3', lw=4)
ax.text(8, 0.55, 'M0', ha='center', color='C3')
# Gregorian M1 at z=85
ax.plot([8.5, 8.5], [-0.8, -0.45], color='C0', lw=4)
ax.plot([8.5, 8.5], [0.45, 0.8], color='C0', lw=4)
ax.text(8.5, 1.0, 'M1', ha='center', color='C0')
# Solar disk light: enters IEO and hits M0 (rejected)
for y in np.linspace(-0.18, 0.18, 5):
    ax.plot([-3, 0, 8], [y * 0.2, y, 0.0], color='gold', lw=1)
    ax.plot([8, 0], [0.0, y], color='gold', lw=0.7, ls=':')
# Coronal light: grazes outside M0
for y in [0.55, -0.55]:
    ax.plot([-3, 0, 8.5], [y * 0.4, y, np.sign(y) * 0.6], color='C2', lw=1.2)
ax.set_xlim(-3.5, 10)
ax.set_ylim(-2.4, 2.4)
ax.set_xlabel('Optical axis z [arbitrary] / 광축')
ax.set_ylabel('Off-axis [arbitrary] / 광축 외')
ax.legend(handles=[plt.Line2D([0], [0], color='gold', label='disk light (rejected) / 디스크 (거부)'),
                   plt.Line2D([0], [0], color='C2', label='coronal light / 코로나 광')],
          loc='lower right', fontsize=9)
ax.set_aspect('equal')

# Right panel: square FoV with annular vignetting.
ax = axes[1]
ax.set_title('Metis FoV at perihelion 0.28 AU / 근일점 시야')
theta = np.linspace(0, 2 * np.pi, 400)
for r_deg, lbl, color in [(1.6, 'inner 1.7 R☉', 'C3'),
                          (2.9, 'outer 3.1 R☉', 'C0'),
                          (3.4, 'corner 3.6 R☉', 'C2')]:
    ax.plot(r_deg * np.cos(theta), r_deg * np.sin(theta), color=color, label=lbl, lw=1.5)
ax.plot([-2.9, 2.9, 2.9, -2.9, -2.9], [-2.9, -2.9, 2.9, 2.9, -2.9],
        color='black', ls='--', lw=1, label='detector edge / 검출기')
ax.set_xlabel('FoV X [deg]')
ax.set_ylabel('FoV Y [deg]')
ax.set_aspect('equal')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.legend(loc='upper right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: VL pB and UV Lyman-α Channel Separation / VL pB와 UV Lyman-α 채널 분리

**English.** Metis splits the two science channels with a tilted (12°) interference filter (Al/MgF₂ multilayer on a MgF₂ substrate) that *transmits* the narrow UV band centred on 121.6 nm and *reflects* the broader VL bandpass 580–640 nm into the polarimeter. The mirrors M0/M1/M2 are coated with the same Al/MgF₂ stack, which sustains high reflectivity in **both** bands. Below we model schematic transmittance/reflectance curves and plot them on the relevant wavelength grid.

**한국어.** Metis는 12° 경사로 설치된 간섭 필터(MgF₂ 기판에 Al/MgF₂ 다층막)로 두 과학 채널을 분리한다: 121.6 nm 중심의 협대역 UV는 *투과*시키고, 580–640 nm의 광대역 VL은 편광계로 *반사*한다. M0/M1/M2 거울도 동일한 Al/MgF₂ 코팅으로 두 파장대 모두에서 높은 반사율을 유지한다. 아래에서 모식적 투과/반사 곡선을 모델링하고 해당 파장 격자에 그린다.

In [ ]:
def filter_transmittance_uv(wavelength_nm):
    """Schematic transmittance of the Metis dichroic in the UV channel.

    Models a Gaussian narrow-band filter peaked at 121.6 nm with FWHM 10 nm.

    Args:
        wavelength_nm: Wavelength [nm] (array_like).

    Returns:
        Transmittance in [0, 1].
    """
    fwhm = 10.0
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    peak = 0.24  # Measured 0.24 +/- 0.01 at 122 nm (paper §4.7)
    return peak * np.exp(-0.5 * ((wavelength_nm - 121.6) / sigma) ** 2)


def filter_reflectance_vl(wavelength_nm):
    """Schematic reflectance of the Metis dichroic in the VL channel.

    Flat-top reflectance over 580--640 nm with smooth roll-off; mean 0.881.

    Args:
        wavelength_nm: Wavelength [nm] (array_like).

    Returns:
        Reflectance in [0, 1].
    """
    edges = (580.0, 640.0)
    centre = 0.5 * sum(edges)
    half = 0.5 * (edges[1] - edges[0])
    # Smooth super-Gaussian flat-top
    return 0.881 * np.exp(-((wavelength_nm - centre) / half) ** 8)


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

wl_uv = np.linspace(110, 135, 600)
axes[0].plot(wl_uv, filter_transmittance_uv(wl_uv), color='C4', lw=2)
axes[0].axvline(121.6, color='black', ls=':', lw=1)
axes[0].set_title('UV channel (transmitted) / UV 채널 (투과)')
axes[0].set_xlabel('Wavelength [nm] / 파장')
axes[0].set_ylabel('Transmittance / 투과율')
axes[0].set_ylim(0, 0.3)
axes[0].grid(alpha=0.3)

wl_vl = np.linspace(500, 720, 600)
axes[1].plot(wl_vl, filter_reflectance_vl(wl_vl), color='C2', lw=2)
axes[1].axvspan(580, 640, color='C2', alpha=0.15, label='580–640 nm bandpass')
axes[1].set_title('VL channel (reflected) / VL 채널 (반사)')
axes[1].set_xlabel('Wavelength [nm] / 파장')
axes[1].set_ylabel('Reflectance / 반사율')
axes[1].set_ylim(0, 1.0)
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**English.** The polarimeter cycles four LCVR retardances (90°, 180°, 270°, 360°) to demodulate the linear Stokes components $Q$ and $U$ from which the polarised brightness $pB = \sqrt{Q^2+U^2}$ is recovered (only the tangential component carries Thomson signal). We simulate one such modulation cycle below.

**한국어.** 편광계는 LCVR의 위상 지연을 4단계(90°/180°/270°/360°)로 순환시켜 선형 Stokes 성분 $Q, U$를 복조하고 $pB = \sqrt{Q^2+U^2}$를 복원한다(접선 성분만 톰슨 산란 신호를 담는다). 아래에서 한 사이클의 변조를 시뮬레이션한다.

In [ ]:
def simulate_polarimeter_cycle(stokes_I, stokes_Q, stokes_U, retardances_deg):
    """Simulate intensities recorded for a sequence of LCVR retardances.

    Idealised analyser at angle 0; LCVR with retardance delta and fast axis at
    +45 degrees produces I_meas = (I + Q cos(delta) + U sin(delta)) / 2.

    Args:
        stokes_I: Total intensity (any positive scalar or array).
        stokes_Q: Linear Q Stokes component.
        stokes_U: Linear U Stokes component.
        retardances_deg: Iterable of LCVR retardances [deg].

    Returns:
        ndarray of measured intensities, one per retardance.
    """
    deltas = np.deg2rad(np.asarray(retardances_deg))
    return 0.5 * (stokes_I + stokes_Q * np.cos(deltas) + stokes_U * np.sin(deltas))


# Tangential polarisation: choose Stokes such that the polarised brightness
# equals 1% of total (typical for the K-corona at 2 R_sun).
I0 = 1.0
pB_true = 0.01
phi_true = np.deg2rad(30.0)  # tangential direction
Q0 = pB_true * np.cos(2 * phi_true)
U0 = pB_true * np.sin(2 * phi_true)

retardances = [90.0, 180.0, 270.0, 360.0]
measured = simulate_polarimeter_cycle(I0, Q0, U0, retardances)

# Demodulate: solve I, Q, U from the four equations (least-squares).
deltas = np.deg2rad(retardances)
design = np.column_stack([np.full_like(deltas, 0.5),
                          0.5 * np.cos(deltas),
                          0.5 * np.sin(deltas)])
I_est, Q_est, U_est = np.linalg.lstsq(design, measured, rcond=None)[0]
pB_recovered = np.hypot(Q_est, U_est)

print(f"Measured intensities for retardances {retardances}: {np.round(measured, 5)}")
print(f"True pB = {pB_true:.5f}; recovered pB = {pB_recovered:.5f}")
print(f"Tangential angle: true {np.rad2deg(phi_true):.2f}°, recovered {0.5*np.rad2deg(np.arctan2(U_est, Q_est)):.2f}°")

## Part 3: Thomson-Scattering Electron Density from pB / pB로부터 톰슨 산란 전자 밀도

**English.** Under spherical symmetry the polarised brightness viewed at impact parameter $\rho$ is
$$pB(\rho) = 2\,K\,\int_\rho^\infty n_e(r)\,\bigl[(1-u)A(r) + u\,B(r)\bigr]\,\frac{\rho^2}{r^2}\,\frac{r\,dr}{\sqrt{r^2-\rho^2}}\;,$$
where $K = \dfrac{B_\odot}{1-u/3}\,\dfrac{3\sigma_T}{16\pi}$ and $A, B$ are the Minnaert (1930) geometric factors with $\sin\gamma = R_\odot/r$. Following van de Hulst (1950), if $pB$ is fitted by a power-law sum $pB(\rho) = \sum_i c_i (\rho/R_\odot)^{-d_i}$ then the electron density follows analytically:
$$n_e(r) = \frac{\sum_i a_i (r/R_\odot)^{-b_i}}{(1-u)A(r) + u\,B(r)},\qquad b_i = d_i + 1,\quad a_i = \frac{c_i}{\sqrt{\pi}\,K}\,\frac{\Gamma((d_i+3)/2)}{\Gamma((d_i+2)/2)}\;.$$

We code the Minnaert factors, compute synthetic $pB$ from a Saito-like double power-law $n_e$ model, fit the power-law expansion, and recover $n_e(r)$ to demonstrate the inversion.

**한국어.** 구대칭 가정 하에서 충돌 매개변수 $\rho$에서 본 편광 밝기는 위 적분식으로 주어지며, $K$ 상수와 Minnaert 인자 $A, B$를 사용한다($\sin\gamma = R_\odot/r$). van de Hulst(1950)에 따라 $pB$를 거듭제곱의 합 $\sum_i c_i (\rho/R_\odot)^{-d_i}$로 적합하면 $n_e(r)$는 위의 닫힌 형태로 해석적 변환된다. 아래에서 Minnaert 인자를 코딩하고, Saito 류 이중 거듭제곱 $n_e$ 모델로부터 합성 $pB$를 만들어 거듭제곱 적합 → 전자 밀도 복원 과정을 시연한다.

In [ ]:
def minnaert_factors(r_rsun):
    """Minnaert (1930) geometric factors A(r) and B(r) for Thomson scattering.

    Args:
        r_rsun: Heliocentric distance [R_sun] (array_like, > 1).

    Returns:
        Tuple (A, B) with the same shape as the input.
    """
    r = np.asarray(r_rsun, dtype=float)
    sin_g = 1.0 / r
    cos_g = np.sqrt(1.0 - sin_g ** 2)
    A = cos_g * sin_g ** 2
    log_term = np.log((1.0 + sin_g) / cos_g)
    B = -(1.0 / 8.0) * (1.0 - 3.0 * sin_g ** 2 - (1.0 + 3.0 * sin_g ** 2) / sin_g * log_term) * cos_g
    return A, B


def K_constant(B_sun=B_SUN, u=U_LIMB):
    """Return the K constant common to pB integrals."""
    return (B_sun / (1.0 - u / 3.0)) * (3.0 * SIGMA_T / (16.0 * np.pi))


def pB_forward(rho_rsun, ne_func, r_max_rsun=20.0, u=U_LIMB):
    """Compute pB(rho) by direct line-of-sight integration of a given n_e(r).

    Args:
        rho_rsun: Impact parameter array [R_sun].
        ne_func: Callable returning n_e(r [R_sun]) in cm^-3.
        r_max_rsun: Outer cutoff [R_sun].
        u: Limb-darkening coefficient.

    Returns:
        pB array in [erg s^-1 cm^-2 sr^-1].
    """
    K = K_constant(u=u)
    pB = np.zeros_like(rho_rsun, dtype=float)
    for i, rho in enumerate(rho_rsun):
        def integrand(r):
            A, B = minnaert_factors(r)
            return (ne_func(r)
                    * ((1 - u) * A + u * B)
                    * (rho ** 2 / r ** 2)
                    * r / np.sqrt(r ** 2 - rho ** 2))
        # convert r from R_sun to cm in the LoS path element
        val, _ = quad(integrand, rho * 1.0001, r_max_rsun, limit=200)
        pB[i] = 2.0 * K * val * R_SUN_CM  # path element dr in cm
    return pB


# Saito-style background-corona electron density (Saito et al. 1977).
def ne_saito(r_rsun):
    """Equatorial Saito 1977 electron density [cm^-3] at heliocentric r [R_sun]."""
    return 1.36e6 * r_rsun ** (-2.14) + 1.68e8 * r_rsun ** (-6.13)


rho_grid = np.linspace(1.7, 6.0, 30)
pB_synth = pB_forward(rho_grid, ne_saito, r_max_rsun=12.0)
print(f"Synthetic pB at rho=2 R_sun: {pB_synth[np.argmin(abs(rho_grid - 2.0))]:.3e} erg/s/cm^2/sr")
print(f"Synthetic pB / B_sun at rho=2 R_sun: {pB_synth[np.argmin(abs(rho_grid - 2.0))] / B_SUN:.2e}")

In [ ]:
def vdh_powerlaw(rho_rsun, c1, d1, c2, d2):
    """Two-term power-law expansion used in the van de Hulst inversion fit."""
    return c1 * rho_rsun ** (-d1) + c2 * rho_rsun ** (-d2)


def van_de_hulst_invert(rho_rsun, pB_obs, u=U_LIMB):
    """Recover n_e(r) from observed pB by a two-term power-law fit + closed form.

    Args:
        rho_rsun: Impact-parameter array [R_sun].
        pB_obs: Observed pB [erg s^-1 cm^-2 sr^-1].
        u: Limb-darkening coefficient.

    Returns:
        Tuple (r_grid, n_e_recovered, fit_params).
    """
    p0 = (pB_obs[0] * rho_rsun[0] ** 3, 3.0, pB_obs[0] * rho_rsun[0] ** 7, 7.0)
    popt, _ = curve_fit(vdh_powerlaw, rho_rsun, pB_obs, p0=p0, maxfev=20000)
    c1, d1, c2, d2 = popt
    K = K_constant(u=u)
    a1 = c1 / (np.sqrt(np.pi) * K) * gamma_fn((d1 + 3) / 2) / gamma_fn((d1 + 2) / 2)
    a2 = c2 / (np.sqrt(np.pi) * K) * gamma_fn((d2 + 3) / 2) / gamma_fn((d2 + 2) / 2)
    b1, b2 = d1 + 1, d2 + 1
    r_grid = np.linspace(rho_rsun.min(), rho_rsun.max(), 100)
    A, B = minnaert_factors(r_grid)
    numerator = a1 * r_grid ** (-b1) + a2 * r_grid ** (-b2)
    # Recovered n_e is in cm^-3 once K carries the proper dimensional prefactor.
    ne = numerator / ((1 - u) * A + u * B) / R_SUN_CM
    return r_grid, ne, popt


r_recov, ne_recov, popt = van_de_hulst_invert(rho_grid, pB_synth)
ne_truth = ne_saito(r_recov)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].loglog(rho_grid, pB_synth, 'o', label='synthetic pB')
axes[0].loglog(rho_grid, vdh_powerlaw(rho_grid, *popt), '-',
               label=f'fit: c1·ρ^-{popt[1]:.2f} + c2·ρ^-{popt[3]:.2f}')
axes[0].set_xlabel('Impact parameter ρ [R☉] / 충돌 매개변수')
axes[0].set_ylabel('pB [erg/s/cm²/sr]')
axes[0].set_title('Power-law fit of pB / pB의 거듭제곱 적합')
axes[0].legend()
axes[0].grid(alpha=0.3, which='both')

axes[1].loglog(r_recov, ne_truth, 'k-', lw=2, label='Saito 1977 (truth)')
axes[1].loglog(r_recov, ne_recov, 'C3--', lw=2, label='vdH inversion')
axes[1].set_xlabel('Heliocentric distance r [R☉] / 일심거리')
axes[1].set_ylabel('n_e [cm⁻³]')
axes[1].set_title('Recovered n_e(r) / 복원된 전자 밀도')
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

rel_err = (ne_recov - ne_truth) / ne_truth
print(f"Mean relative error in recovered n_e: {np.mean(np.abs(rel_err)) * 100:.2f}%")
print(f"Max  relative error in recovered n_e: {np.max(np.abs(rel_err)) * 100:.2f}%")

## Part 4: Doppler-Dimming Inversion for Outflow Speed / 유출 속도의 도플러 디밍 역해

**English.** Coronal H I atoms resonantly scatter chromospheric Lyman-α photons. If the corona moves outward at speed $w$ along the incident-radiation direction $\mathbf{n}$, the absorption profile is shifted by $\delta\lambda = (\lambda_0/c)\,\mathbf{w}\!\cdot\!\mathbf{n}$, reducing the overlap with the chromospheric emission profile and *dimming* the resonantly scattered intensity. The dimming integral
$$\Phi(\delta\lambda) = \int_0^\infty I_\text{ex}(\lambda - \delta\lambda)\,\Psi(\lambda - \lambda_0)\,d\lambda$$
is monotonic in $w$ for $0 \lesssim w \lesssim 500\,\mathrm{km/s}$; inverting $\Phi_\text{meas}/\Phi_\text{static}$ yields $w$. Below we use Gaussian profiles for both the chromospheric source $I_\text{ex}$ (FWHM ≈ 0.7 Å) and the coronal absorber $\Psi$ (thermal width set by $T_{k}$), tabulate $\Phi(w)/\Phi(0)$ vs $w$, and use it to recover an injected outflow speed.

**한국어.** 코로나의 H I 원자는 색구의 Lyman-α 광자를 공명 산란한다. 코로나가 입사 방향 $\mathbf{n}$을 따라 속도 $w$로 유출되면 흡수 프로파일이 $\delta\lambda = (\lambda_0/c)\,\mathbf{w}\!\cdot\!\mathbf{n}$만큼 이동해 색구 방출 프로파일과의 중첩이 감소하고, 산란된 강도가 *디밍*된다. 디밍 적분 $\Phi(\delta\lambda)$는 $0 \lesssim w \lesssim 500$ km/s에서 단조 감소이므로 $\Phi_\text{meas}/\Phi_\text{static}$의 역해로 $w$를 얻는다. 아래에서 색구 광원 $I_\text{ex}$(FWHM ≈ 0.7 Å)와 코로나 흡수체 $\Psi$(열 폭은 $T_{k}$로 결정)를 모두 가우시안으로 두고 $\Phi(w)/\Phi(0)$를 표화한 뒤, 주어진 유출 속도를 회수한다.

In [ ]:
def gaussian(x, mu, sigma):
    """Normalised Gaussian profile."""
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma)


def coronal_absorber_width(T_k):
    """Thermal Gaussian sigma of the coronal Lyman-α absorber [angstrom].

    Args:
        T_k: Kinetic temperature of H atoms [K].

    Returns:
        Sigma in angstroms.
    """
    v_th_kms = np.sqrt(K_B * T_k / M_P) * 1e-5  # km/s
    return LAMBDA_LYA * v_th_kms / C_LIGHT


def doppler_dimming_factor(w_kms, T_k, sigma_ex_AA=0.30):
    """Compute the Doppler-dimming factor Phi(w)/Phi(0).

    Models the chromospheric exciting profile as a Gaussian (sigma_ex_AA)
    centred at lambda_0 and the coronal absorber as a Gaussian shifted by
    delta_lambda = (lambda_0/c) * w. Returns the overlap integral
    normalised by its w=0 value.

    Args:
        w_kms: Outflow speed [km/s] (scalar).
        T_k: H I kinetic temperature [K].
        sigma_ex_AA: Chromospheric Lyman-α profile sigma [angstrom].

    Returns:
        Dimming factor in [0, 1].
    """
    sigma_a = coronal_absorber_width(T_k)
    sigma_eff = np.sqrt(sigma_ex_AA ** 2 + sigma_a ** 2)
    delta_lambda = LAMBDA_LYA * w_kms / C_LIGHT
    # Gaussian-Gaussian overlap (closed form): exp(-d^2 / (2 * sigma_eff^2))
    # divided by its w=0 value (which is 1).
    return np.exp(-0.5 * (delta_lambda / sigma_eff) ** 2)


T_kin = 1.5e6  # K — typical H kinetic temperature in equatorial streamers
w_grid = np.linspace(0, 500, 200)
phi_grid = np.array([doppler_dimming_factor(w, T_kin) for w in w_grid])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(w_grid, phi_grid, lw=2)
ax.set_xlabel('Outflow speed w [km/s] / 유출 속도')
ax.set_ylabel('Φ(w) / Φ(0)  (Doppler-dimming factor)')
ax.set_title('Doppler-dimming curve at T_k = 1.5×10⁶ K / 도플러 디밍 곡선')
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.axvspan(80, 500, color='C2', alpha=0.1, label='Metis sensitivity 80–500 km/s')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def invert_doppler_dimming(phi_obs, T_k, sigma_ex_AA=0.30, w_max=600.0):
    """Solve Phi(w)/Phi(0) = phi_obs for w using Brent root finding.

    Args:
        phi_obs: Measured dimming factor (between 0 and 1).
        T_k: H I kinetic temperature [K].
        sigma_ex_AA: Source profile sigma [angstrom].
        w_max: Upper bracket for outflow speed [km/s].

    Returns:
        Recovered outflow speed [km/s].
    """
    return brentq(lambda w: doppler_dimming_factor(w, T_k, sigma_ex_AA) - phi_obs,
                  0.0, w_max)


rng = np.random.default_rng(42)
true_speeds = np.array([50, 100, 200, 350, 500])
phi_meas = np.array([doppler_dimming_factor(w, T_kin) for w in true_speeds])
phi_meas_noisy = phi_meas * (1 + 0.02 * rng.standard_normal(true_speeds.size))
recovered = np.array([invert_doppler_dimming(p, T_kin) for p in phi_meas_noisy])

print(f"{'true w [km/s]':>14} | {'phi (noisy)':>12} | {'recovered w [km/s]':>20}")
print('-' * 56)
for wt, p, wr in zip(true_speeds, phi_meas_noisy, recovered):
    print(f"{wt:>14.1f} | {p:>12.4f} | {wr:>20.2f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([0, 550], [0, 550], 'k:', lw=1, label='ideal')
ax.errorbar(true_speeds, recovered, yerr=10, fmt='o', color='C3', label='recovered')
ax.set_xlabel('True outflow speed [km/s] / 진 유출 속도')
ax.set_ylabel('Recovered speed [km/s] / 복원 속도')
ax.set_title('Doppler-dimming inversion accuracy / 도플러 디밍 역해 정확도')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

**English.** We exercised four pillars of the Metis science chain on synthetic data: (1) the IEO geometry and FoV scaling with heliocentric distance, (2) UV/VL channel separation by the tilted Al/MgF₂ multilayer plus four-step LCVR demodulation of $pB$, (3) van de Hulst inversion of $pB(\rho)$ to recover the Saito (1977) $n_e(r)$ within a few percent, and (4) Doppler-dimming inversion that returns injected H outflow speeds 50–500 km/s with sub-5% bias.

**한국어.** 합성 데이터로 Metis 과학 사슬의 네 기둥을 시연했다: (1) IEO 기하 및 일심거리에 따른 시야 스케일링, (2) 경사 Al/MgF₂ 다층막에 의한 UV/VL 분리와 4단계 LCVR 변조로 $pB$ 복조, (3) van de Hulst 역해로 $pB(\rho)$ → Saito (1977) $n_e(r)$를 수% 이내로 복원, (4) 도플러 디밍 역해로 50–500 km/s 유출 속도를 5% 이내 오차로 회수.

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| External occulter / 외부 차폐 | IEO 40 mm circular aperture | LASCO C2/C3 disk; PROBA-3 ASPIICS formation-flying disk |
| pB → n_e inversion / pB 역해 | van de Hulst (1950) closed form | Numerical regularised Abel inversion (e.g. PyAbel, Frazin tomography) |
| Doppler dimming / 도플러 디밍 | Lyman-α from chromosphere | UVCS (1996–2012); future generations on ASO-S LST and Aditya-L1 SUIT |
| VL+UV simultaneous / VL+UV 동시 | Al/MgF₂ multilayer beam-splitter | ASO-S LST (Lyα + WL); future PROBA-3 ASPIICS |
| Onboard compression / 탑재 압축 | Radialised CCSDS-123 (≤51×) | JPEG-2000, ZFP for in-situ heliophysics |